# Stage 3b — Fine-tuned BERTimbau baseline

Full fine-tuning of `neuralmind/bert-base-portuguese-cased` (110M parameters) on the B2W
training pool, evaluated on RePro **dev** and compared with the best TF-IDF baseline.
Decisions: D14 in `docs/decisions.md`.

Runs on **Google Colab** or **Kaggle**; the first code cell detects which one.

- **Colab:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
- **Kaggle:** in *Settings*, set *Accelerator → GPU T4 x2* and turn *Internet* on, then
  *Save Version → Save & Run All*. The zip appears in the version's *Output* tab.
  Avoid the P100: recent PyTorch builds no longer support its architecture.

What this notebook does:

1. Clones the repository and rebuilds the data, checking it matches the committed summary.
2. Measures review lengths in tokens and picks `MAX_LEN` (smallest of 128/256/384/512 covering 95%).
3. Trains for 2 epochs with class-weighted loss and keeps the epoch with the best dev macro-F1.
4. Evaluates on dev (macro-F1, 95% bootstrap CI, paired bootstrap against TF-IDF).
5. Predicts the **test** set but computes **no** test metrics (decision D12: test is scored once, in Stage 6).
6. Packs the dev report and the prediction files into `bertimbau_results.zip`
   (downloaded by the browser on Colab; saved to `/kaggle/working` on Kaggle).

Training time on a T4 is not known yet; it is measured and saved in the report.
GPU training is not bit-for-bit deterministic, so a rerun can give slightly different numbers.

In [ ]:
import importlib.util
import os
import subprocess
import sys

import torch

# Where are we? Kaggle sets KAGGLE_KERNEL_RUN_TYPE; Colab ships the google.colab package.
if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    PLATFORM = "kaggle"
    WORK_DIR = "/tmp"                # scratch space: the clone and data are not kept as output
    OUTPUT_DIR = "/kaggle/working"   # only files here are saved with the notebook version
elif importlib.util.find_spec("google.colab") is not None:
    PLATFORM = "colab"
    WORK_DIR = OUTPUT_DIR = "/content"
else:
    raise RuntimeError("Run this notebook on Google Colab or Kaggle")
print("platform:", PLATFORM)

assert torch.cuda.is_available(), (
    "No GPU. Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2"
)
GPU_NAME = torch.cuda.get_device_name(0)
print(GPU_NAME)

REPO_URL = "https://github.com/PedroRicci12/ptbr-review-lora.git"
REPO_DIR = f"{WORK_DIR}/ptbr-review-lora"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")
REPO_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("repository commit:", REPO_COMMIT)

## 1. Data

Same script as on the laptop. If the counts differ from the committed
`reports/data_summary.json` (for example, because of a different pandas version), the cell stops.

In [ ]:
subprocess.run(
    [sys.executable, "scripts/prepare_data.py"],
    check=True, stdout=subprocess.DEVNULL, env={**os.environ, "PYTHONPATH": "src"},
)
diff = subprocess.run(
    ["git", "diff", "--exit-code", "reports/data_summary.json"], capture_output=True, text=True
)
assert diff.returncode == 0, "Prepared data differs from the repository:\n" + diff.stdout
print("data_summary.json identical to the repository")

## 2. Configuration (fixed, not tuned)

In [ ]:
MODEL_ID = "neuralmind/bert-base-portuguese-cased"
MODEL_REVISION = "94d69c95f98f7d5b2a8700c420230ae10def0baa"  # pinned (weights are a pickle file)

SEED = 42
EPOCHS = 2
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
USE_TITLE = True           # chosen on dev for TF-IDF (D13)
CLASS_WEIGHT = "balanced"  # D13
COVERAGE = 0.95
CANDIDATE_LENGTHS = (128, 256, 384, 512)
N_RESAMPLES = 1000
TFIDF_REFERENCE = "tfidf_lr__title_text__balanced"

import random

import numpy as np

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
from pathlib import Path

import pandas as pd

from ptbr_review_lora.data import LABELS, model_input

pool = pd.read_parquet("data/processed/b2w_pool.parquet")
dev = pd.read_parquet("data/processed/repro_dev.parquet")
test = pd.read_parquet("data/processed/repro_test.parquet")

LABEL2ID = {label: i for i, label in enumerate(LABELS)}
train_texts = model_input(pool, USE_TITLE).tolist()
train_labels = pool["label"].map(LABEL2ID).tolist()
dev_texts = model_input(dev, USE_TITLE).tolist()
test_texts = model_input(test, USE_TITLE).tolist()
y_dev = dev["label"].tolist()
print(len(train_texts), len(dev_texts), len(test_texts))

## 3. Review length in tokens → `MAX_LEN`

Longer inputs cost more memory and time; texts longer than `MAX_LEN` are truncated.
Measured on a random sample of 20,000 training reviews.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)

sample = pd.Series(train_texts).sample(20_000, random_state=SEED).tolist()
lengths = np.array([len(ids) for ids in tokenizer(sample)["input_ids"]])
token_percentiles = {p: int(np.percentile(lengths, p)) for p in (50, 90, 95, 99)}
coverage_by_max_len = {length: float(np.mean(lengths <= length)) for length in CANDIDATE_LENGTHS}
MAX_LEN = next((L for L in CANDIDATE_LENGTHS if coverage_by_max_len[L] >= COVERAGE), 512)

print("percentiles (tokens):", token_percentiles)
print("coverage:", {k: round(v, 3) for k, v in coverage_by_max_len.items()})
print("MAX_LEN =", MAX_LEN)

## 4. Model, data loaders, optimizer

- Texts are tokenized once; each batch is padded only to its longest text (dynamic padding).
- Class weights make an error on the rare `neutro` class cost more than one on `positivo`.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)


def encode(texts, labels=None):
    enc = tokenizer(texts, truncation=True, max_length=MAX_LEN)
    items = [{"input_ids": ids, "attention_mask": mask} for ids, mask in zip(enc["input_ids"], enc["attention_mask"])]
    if labels is not None:
        for item, label in zip(items, labels):
            item["labels"] = label
    return items


collator = DataCollatorWithPadding(tokenizer)
train_loader = DataLoader(
    encode(train_texts, train_labels), batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collator, generator=torch.Generator().manual_seed(SEED),
)
dev_loader = DataLoader(encode(dev_texts), batch_size=128, collate_fn=collator)
test_loader = DataLoader(encode(test_texts), batch_size=128, collate_fn=collator)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, num_labels=len(LABELS),
    id2label=dict(enumerate(LABELS)), label2id=LABEL2ID,
).cuda()

class_weights = compute_class_weight(CLASS_WEIGHT, classes=np.arange(len(LABELS)), y=np.array(train_labels))
loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float).cuda())
print("class weights:", dict(zip(LABELS, class_weights.round(3))))

# Standard BERT recipe: no weight decay on biases and LayerNorm weights.
no_decay = ("bias", "LayerNorm.weight")
optimizer = torch.optim.AdamW(
    [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
    ],
    lr=LEARNING_RATE,
)
total_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda")  # mixed precision (fp16): faster and lighter on a T4
print("training steps:", total_steps)

## 5. Training (keeps the epoch with the best dev macro-F1)

In [ ]:
import time

from tqdm.auto import tqdm

from ptbr_review_lora.metrics import macro_f1


@torch.no_grad()
def predict(loader):
    model.eval()
    ids = []
    for batch in loader:
        batch = {k: v.cuda() for k, v in batch.items()}
        with torch.autocast("cuda", dtype=torch.float16):
            ids.extend(model(**batch).logits.argmax(-1).tolist())
    return [LABELS[i] for i in ids]


history = []
best_f1, best_epoch, best_state = -1.0, None, None
start = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for batch in tqdm(train_loader, desc=f"epoch {epoch}"):
        batch = {k: v.cuda() for k, v in batch.items()}
        labels = batch.pop("labels")
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**batch).logits
        loss = loss_fn(logits.float(), labels)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item()

    dev_f1 = macro_f1(y_dev, predict(dev_loader))
    history.append({
        "epoch": epoch,
        "train_loss": running_loss / len(train_loader),
        "dev_macro_f1": dev_f1,
        "elapsed_seconds": round(time.perf_counter() - start),
    })
    print(history[-1])
    if dev_f1 > best_f1:
        best_f1, best_epoch = dev_f1, epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

TRAIN_SECONDS = round(time.perf_counter() - start)
model.load_state_dict(best_state)
print(f"best epoch: {best_epoch} (dev macro-F1 {best_f1:.4f}), training time {TRAIN_SECONDS / 60:.1f} min")

## 6. Evaluation on dev and comparison with TF-IDF

The TF-IDF dev predictions come from `predictions/dev/` in the repository, so both models are
compared on exactly the same reviews with the paired bootstrap (decision D11).

In [ ]:
from ptbr_review_lora.metrics import (
    bootstrap_ci,
    compute_metrics,
    neutral_recall_by_group,
    paired_bootstrap,
)
from ptbr_review_lora.predictions import load_predictions, save_predictions

bert_dev = predict(dev_loader)
_, low, high = bootstrap_ci(y_dev, bert_dev, n_resamples=N_RESAMPLES, seed=SEED)
dev_results = {
    **compute_metrics(y_dev, bert_dev),
    "macro_f1_ci95": [low, high],
    "neutral_recall_by_group": neutral_recall_by_group(dev["polarity_group"], bert_dev),
}

tfidf_dev = load_predictions(Path(f"predictions/dev/{TFIDF_REFERENCE}.csv"), len(dev))
comparison = paired_bootstrap(y_dev, tfidf_dev, bert_dev, n_resamples=N_RESAMPLES, seed=SEED)

print(f"BERTimbau dev macro-F1: {dev_results['macro_f1']:.4f} (95% CI {low:.4f}-{high:.4f})")
print(f"accuracy: {dev_results['accuracy']:.4f}")
print("F1 per class:", {k: round(v, 3) for k, v in dev_results["f1_per_class"].items()})
print("neutral recall:", {k: round(v, 3) for k, v in dev_results["neutral_recall_by_group"].items()})
print(f"vs TF-IDF: diff {comparison['diff']:+.4f} (95% CI {comparison['low']:+.4f} to {comparison['high']:+.4f}), "
      f"p_b_not_better {comparison['p_b_not_better']:.3f}")

## 7. Test predictions (no metrics)

Decision D12: the test set is scored once, in Stage 6, for all frozen models together.
Saving the predictions now avoids retraining later; committing them before Stage 6 also shows
they were not changed after seeing test results.

In [ ]:
bert_test = predict(test_loader)
save_predictions(Path("predictions/dev/bertimbau.csv"), bert_dev)
save_predictions(Path("predictions/test/bertimbau.csv"), bert_test)
print(len(bert_dev), len(bert_test))

## 8. Report and download

In [ ]:
import json
import platform

import sklearn
import transformers

report = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "repo_commit": REPO_COMMIT,
    "environment": {
        "platform": PLATFORM,
        "gpu": GPU_NAME,
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "scikit_learn": sklearn.__version__,
        "pandas": pd.__version__,
    },
    "hyperparameters": {
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
        "warmup_ratio": WARMUP_RATIO, "weight_decay": WEIGHT_DECAY, "max_grad_norm": MAX_GRAD_NORM,
        "use_title": USE_TITLE, "class_weight": CLASS_WEIGHT, "fp16": True, "seed": SEED,
    },
    "class_weights": dict(zip(LABELS, class_weights.tolist())),
    "max_len": MAX_LEN,
    "token_length_percentiles": token_percentiles,
    "coverage_by_max_len": coverage_by_max_len,
    "history": history,
    "best_epoch": best_epoch,
    "train_seconds": TRAIN_SECONDS,
    "split": "repro_dev",
    "n": len(dev),
    "n_resamples": N_RESAMPLES,
    "results": {"bertimbau": dev_results},
    "paired_comparisons": {f"bertimbau vs {TFIDF_REFERENCE}": comparison},
}
Path("reports/bertimbau_dev.json").write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

# Python's zipfile instead of the `zip` command, which is not guaranteed to be installed.
import zipfile

ZIP_PATH = f"{OUTPUT_DIR}/bertimbau_results.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in ("reports/bertimbau_dev.json", "predictions/dev/bertimbau.csv", "predictions/test/bertimbau.csv"):
        zf.write(path)  # relative paths, so the files unpack into the same folders of the repository
print("saved", ZIP_PATH)

if PLATFORM == "colab":
    from google.colab import files

    files.download(ZIP_PATH)
else:
    print("Kaggle: download it from the Output tab of this notebook version")

## Next step

Hand `bertimbau_results.zip` over: its files go into the repository at the same paths
(`reports/` and `predictions/`). The model weights are not kept; the test predictions are all
that Stage 6 needs.